# Scraped data → training CSV → retrain churn model → graphics

This notebook:

1. **Scrapes** public FDIC + Wikipedia pages (same script as `scraping/scrape_public_sources.py`).
2. **Builds** `data/customers_augmented_public_health.csv` by merging a **state→macro-region** uninsured-rate proxy onto every synthetic customer row (`scraping/build_augmented_customers_from_scraped.py`).
3. **Explores** the scraped tables and the augmented customer file with charts **before** fitting.
4. **Retrains** the churn classifier (same recipe as notebook 02: preprocessor + gradient boosting + isotonic calibration) on the augmented CSV.
5. **Reports** how many rows were used and **accuracy / ROC-AUC**, and saves **ROC, PR, and confusion-matrix** figures under `outputs/`.

In [ ]:
import subprocess
import sys
from pathlib import Path

%matplotlib inline

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

for script in [
    ROOT / "scraping" / "scrape_public_sources.py",
    ROOT / "scraping" / "build_augmented_customers_from_scraped.py",
]:
    print("Running", script.name, "...")
    subprocess.run([sys.executable, str(script)], cwd=str(ROOT), check=True)
print("Scrape + build complete.")

## Scraped-source tables (quick visuals)

Before touching the customer file, inspect what came from the web.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

fdic = pd.read_csv(ROOT / "data" / "scraped" / "fdic_failed_banks_list.csv")
unin = pd.read_csv(ROOT / "data" / "scraped" / "wikipedia_us_uninsured_by_year.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
top_states = fdic["State"].value_counts().head(12)
sns.barplot(x=top_states.values, y=top_states.index, ax=axes[0])
axes[0].set_title("FDIC failed banks in scrape (top states)")

year_col = "Year"
pct_col = "Uninsured Percent"
pct_num = unin[pct_col].astype(str).str.rstrip("%").astype(float)
axes[1].bar(range(len(unin)), pct_num, tick_label=unin[year_col].astype(str))
axes[1].set_xticklabels(unin[year_col].astype(str), rotation=45, ha="right")
axes[1].set_title("U.S. uninsured % (Wikipedia table)")
axes[1].set_xlabel("Year label (as published)")
axes[1].set_ylabel("Uninsured %")
plt.tight_layout()
plt.show()

print("FDIC rows in scrape:", len(fdic), "| uninsured timeline rows:", len(unin))

## Augmented customers — explore before training

The proxy column is **constant within each macro-region** (it is the mean uninsured % across states in that region, averaged across years in the scraped table). It still gives the model a **public-context** signal before we fit churn.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

CFG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
path = ROOT / str(CFG["data"]["customer_training_csv"])
df = pd.read_csv(path)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["public_uninsured_rate_proxy"], kde=True, ax=axes[0])
axes[0].set_title("Distribution of public uninsured proxy")

sns.boxplot(data=df, x="churned", y="public_uninsured_rate_proxy", ax=axes[1])
axes[1].set_title("Proxy vs churn label")
plt.tight_layout()
plt.show()

print(path)
print(df.shape)

## Retrain churn model + graphical metrics

Fit on **training rows only**, evaluate on the **held-out test** set, and save figures to `outputs/`.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES: List[str] = [
    "age", "tenure_months", "num_orders", "total_spend", "avg_order_value",
    "days_since_last_order", "email_opens_30d", "app_sessions_30d",
    "public_uninsured_rate_proxy",
]
CATEGORICAL_FEATURES: List[str] = ["region"]
TARGET_COLUMN = "churned"
INTERNAL_COLUMNS: Tuple[str, ...] = ("_acquisition_prob", "_monthly_spend_intensity")

@dataclass
class TrainValTestSplit:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    acquisition_prob_train: np.ndarray
    acquisition_prob_val: np.ndarray
    acquisition_prob_test: np.ndarray

def make_preprocessor() -> ColumnTransformer:
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])

def temporal_or_random_split(df, train_fraction, val_fraction, random_seed):
    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be in (0,1)")
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0,1)")
    if train_fraction + val_fraction >= 1:
        raise ValueError("train_fraction + val_fraction must be < 1")
    work = df.reset_index(drop=True)
    idx = np.arange(len(work))
    rng = np.random.default_rng(random_seed)
    rng.shuffle(idx)
    n = len(idx)
    n_train = int(n * train_fraction)
    n_val = int(n * val_fraction)
    train_idx = idx[:n_train]
    val_idx = idx[n_train : n_train + n_val]
    test_idx = idx[n_train + n_val :]

    def _pack(indices):
        part = work.iloc[indices]
        y = part[TARGET_COLUMN].to_numpy()
        latent = part["_acquisition_prob"].to_numpy()
        drop_cols = list(INTERNAL_COLUMNS) + [TARGET_COLUMN]
        if "customer_id" in part.columns:
            drop_cols.append("customer_id")
        X = part.drop(columns=drop_cols)
        return X, y, latent

    X_train, y_train, a_train = _pack(train_idx)
    X_val, y_val, a_val = _pack(val_idx)
    X_test, y_test, a_test = _pack(test_idx)
    return TrainValTestSplit(
        X_train=X_train, X_val=X_val, X_test=X_test,
        y_train=y_train, y_val=y_val, y_test=y_test,
        acquisition_prob_train=a_train, acquisition_prob_val=a_val, acquisition_prob_test=a_test,
    )

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

CFG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
SEED = int(CFG["project"]["random_seed"])
OUT = ROOT / str(CFG["project"]["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)

path = ROOT / str(CFG["data"]["customer_training_csv"])
df = pd.read_csv(path)
split = temporal_or_random_split(
    df,
    train_fraction=float(CFG["data"]["train_fraction"]),
    val_fraction=float(CFG["data"]["val_fraction"]),
    random_seed=SEED,
)

cm = CFG["churn_model"]
clf = GradientBoostingClassifier(
    random_state=SEED,
    n_estimators=int(cm["n_estimators"]),
    max_depth=int(cm["max_depth"]),
    learning_rate=float(cm["learning_rate"]),
    min_samples_leaf=int(cm["min_samples_leaf"]),
)
base = Pipeline(steps=[("prep", make_preprocessor()), ("clf", clf)])
model = CalibratedClassifierCV(estimator=base, method="isotonic", cv=3)
model.fit(split.X_train, split.y_train)

y_score = model.predict_proba(split.X_test)[:, 1]
y_pred = (y_score >= 0.5).astype(int)

acc = float(accuracy_score(split.y_test, y_pred))
roc = float(roc_auc_score(split.y_test, y_score))
pr_auc = float(average_precision_score(split.y_test, y_score))

print("=== Training data (rows) ===")
print("Train:", len(split.y_train), "| Val:", len(split.y_val), "| Test:", len(split.y_test))
print("Total customers in file:", len(df))
print()
print("=== Test-set metrics (churn) ===")
print(f"Accuracy (threshold 0.5): {acc:.4f}")
print(f"ROC-AUC: {roc:.4f}")
print(f"PR-AUC (average precision): {pr_auc:.4f}")

fig, ax = plt.subplots(figsize=(4.2, 4))
ConfusionMatrixDisplay.from_predictions(split.y_test, y_pred, ax=ax, colorbar=False)
ax.set_title("Confusion matrix (test)")
fig.tight_layout()
fig.savefig(OUT / "scraped_augment_confusion_matrix.png", dpi=150)
plt.close(fig)

fpr, tpr, _ = roc_curve(split.y_test, y_score)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(fpr, tpr, label=f"ROC-AUC = {roc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curve (test)")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "scraped_augment_roc.png", dpi=150)
plt.close(fig)

prec, rec, _ = precision_recall_curve(split.y_test, y_score)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(rec, prec)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision–recall curve (PR-AUC={pr_auc:.3f})")
fig.tight_layout()
fig.savefig(OUT / "scraped_augment_pr.png", dpi=150)
plt.close(fig)

payload = {
    "training_csv": str(path.relative_to(ROOT)),
    "n_total_rows": int(len(df)),
    "n_train": int(len(split.y_train)),
    "n_val": int(len(split.y_val)),
    "n_test": int(len(split.y_test)),
    "test_accuracy": acc,
    "test_roc_auc": roc,
    "test_pr_auc": pr_auc,
    "figures": [
        "scraped_augment_confusion_matrix.png",
        "scraped_augment_roc.png",
        "scraped_augment_pr.png",
    ],
}
(OUT / "scraped_retrain_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote metrics + figures to", OUT)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
OUT = ROOT / "outputs"
for name in ["scraped_augment_confusion_matrix.png", "scraped_augment_roc.png", "scraped_augment_pr.png"]:
    display(Image(filename=str(OUT / name)))